# Post Vector DB — ChromaDB Three-Partition System

Three ChromaDB collections mirror the hot / warm / cold retrieval architecture:

| Collection | Age window | Categories allowed |
|---|---|---|
| `posts_hot` | 0 – 72 hrs | all four |
| `posts_warm` | 72 hrs – 5 days | deal_req · discussion · knowledge |
| `posts_cold` | 5 – 30 days | knowledge only |

**How `expires_at` is set at post creation:**  
The moment a user hits Publish, the backend computes:  
```
expires_at = created_at + CATEGORY_EXPIRY[category]
```
where `CATEGORY_EXPIRY = {market_update: 2d, deal_req: 7d, discussion: 14d, knowledge: 90d}`.  
This value travels with the vector as a ChromaDB metadata field and drives every expiry decision downstream — no separate expiry service needed.


## Cell 01 — Imports & config

In [1]:
import math, json, uuid, warnings
import numpy as np
import pandas as pd
import chromadb
from chromadb.config import Settings
from datetime import datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Cutoff = Apr 3 2026 10:00 AM UTC (matches dataset)
CUTOFF = datetime(2026, 4, 3, 10, 0, 0, tzinfo=timezone.utc)

# ── Partition boundaries
HOT_HOURS  = 72
WARM_DAYS  = 5
COLD_DAYS  = 30

# ── Category expiry at post-creation time (days)
CATEGORY_EXPIRY = {
    "market_update": 2,
    "deal_req":      7,
    "discussion":    14,
    "knowledge":     90,
}

# ── Which partitions each category can live in
CATEGORY_PARTITIONS = {
    "market_update": ["hot"],
    "deal_req":      ["hot", "warm"],
    "discussion":    ["hot", "warm"],
    "knowledge":     ["hot", "warm", "cold"],
}

# ── Vector layout  (18 dims total)
COMMODITIES = ["rice","wheat","cotton","sugar","soybean",
               "maize","spices","pulses","groundnut","mustard"]
ROLES       = ["trader","exporter","broker"]
VEC_SIZE    = 18   # 10 commodity + 3 role + 3 geo + 2 qty

CSV_PATH    = "posts_vector_db.csv"   # update path if needed
CHROMA_PATH = "./chroma_post_db"      # local persistent directory

print("Config ready.")
print(f"  Cutoff : {CUTOFF.isoformat()}")
print(f"  Vector : {VEC_SIZE} dims  ({len(COMMODITIES)} commodity + {len(ROLES)} role + 3 geo + 2 qty)")


Config ready.
  Cutoff : 2026-04-03T10:00:00+00:00
  Vector : 18 dims  (10 commodity + 3 role + 3 geo + 2 qty)


In [3]:
!pip uninstall chromadb-client -y
!pip install chromadb

Found existing installation: chromadb-client 1.5.5
Uninstalling chromadb-client-1.5.5:
  Successfully uninstalled chromadb-client-1.5.5


## Cell 02 — ChromaDB client + three collections

In [5]:
# Persistent client — data survives between sessions
client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False),
)

def get_or_create(name):
    """
    Get collection if exists, else create.
    ChromaDB uses cosine distance natively — matches our cosine similarity.
    """
    return client.get_or_create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"},   # cosine distance index
    )

posts_hot  = get_or_create("posts_hot")
posts_warm = get_or_create("posts_warm")
posts_cold = get_or_create("posts_cold")

COLLECTIONS = {
    "hot":  posts_hot,
    "warm": posts_warm,
    "cold": posts_cold,
}

print("Collections ready:")
print(f"  posts_hot  : {posts_hot.count()} vectors")
print(f"  posts_warm : {posts_warm.count()} vectors")
print(f"  posts_cold : {posts_cold.count()} vectors")


RuntimeError: Chroma is running in http-only client mode, and can only be run with 'chromadb.api.fastapi.FastAPI' or 'chromadb.api.async_fastapi.AsyncFastAPI' as the chroma_api_impl.             see https://docs.trychroma.com/guides#using-the-python-http-only-client for more information.

## Cell 03 — Vector builder (18 dims)

In [ ]:
def build_vector(commodity, target_roles_str, lat, lon,
                 category, qty_min=None, qty_max=None):
    """
    Build an 18-dim float vector from post (or user profile) fields.

    Layout:
        [0:10]   commodity one-hot
        [10:13]  role multi-hot  (target_roles pipe-separated)
        [13:16]  geo 3D cartesian (lat/lon → unit sphere)
        [16:18]  qty normalised 0-1  (zeros for non-deal categories)

    Same function is used at query time for the USER profile vector.
    """
    # ── commodity one-hot
    com = [1.0 if c == commodity else 0.0 for c in COMMODITIES]

    # ── role multi-hot
    roles_set = set(str(target_roles_str).split("|"))
    rol = [1.0 if r in roles_set else 0.0 for r in ROLES]

    # ── geo 3D cartesian
    lat_r, lon_r = math.radians(float(lat)), math.radians(float(lon))
    geo = [
        round(math.cos(lat_r) * math.cos(lon_r), 6),
        round(math.cos(lat_r) * math.sin(lon_r), 6),
        round(math.sin(lat_r), 6),
    ]

    # ── qty normalised (0-5000 MT scale; zeros for non-deal posts)
    if category == "deal_req" and qty_min is not None and not (
        isinstance(qty_min, float) and math.isnan(qty_min)
    ):
        qty = [
            round(min(float(qty_min) / 5000.0, 1.0), 4),
            round(min(float(qty_max) / 5000.0, 1.0), 4),
        ]
    else:
        qty = [0.0, 0.0]

    return com + rol + geo + qty

# Quick sanity check
test_vec = build_vector("rice", "trader|broker", 19.076, 72.877, "deal_req", 50, 200)
assert len(test_vec) == VEC_SIZE
print(f"Vector dims : {len(test_vec)}  ✓")
print(f"Sample      : {test_vec}")


## Cell 04 — Partition assignment logic

In [ ]:
def assign_partition(created_at_str, category):
    """
    Returns the correct collection name for a post based on:
      1. Its age relative to CUTOFF
      2. Whether its category is allowed in that partition

    Returns None if the post is too old / category not allowed.
    """
    created_at = pd.to_datetime(created_at_str, utc=True)
    age_hours  = (CUTOFF - created_at).total_seconds() / 3600

    if age_hours <= HOT_HOURS:
        partition = "hot"
    elif age_hours <= WARM_DAYS * 24:
        partition = "warm"
    elif age_hours <= COLD_DAYS * 24:
        partition = "cold"
    else:
        return None   # older than 30 days — don't insert

    allowed = CATEGORY_PARTITIONS.get(category, ["hot","warm","cold"])
    if partition not in allowed:
        return None   # category expired by this age (e.g. market_update in warm)

    return partition

# ── Verify partition rules
test_cases = [
    ("2026-04-03 08:00:00+00:00", "market_update"),  # 2h ago → hot
    ("2026-04-01 10:00:00+00:00", "market_update"),  # 48h ago → None (market expires at 2d)
    ("2026-04-01 10:00:00+00:00", "deal_req"),        # 48h ago → warm
    ("2026-03-29 10:00:00+00:00", "deal_req"),        # 5d ago → None (deal expires at 7d, but >5d goes cold, not allowed)
    ("2026-03-29 10:00:00+00:00", "knowledge"),       # 5d ago → cold
    ("2026-03-01 10:00:00+00:00", "knowledge"),       # 33d ago → None (>30d)
]

print(f"{'created_at':<30} {'category':<15} {'→ partition'}")
print("─" * 65)
for ts, cat in test_cases:
    result = assign_partition(ts, cat)
    print(f"{ts:<30} {cat:<15}  {result or 'SKIP (expired/not allowed)'}")


## Cell 05 — Load CSV into three ChromaDB collections

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"CSV loaded: {len(df):,} rows × {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")


In [ ]:
BATCH_SIZE = 500

counts  = {"hot": 0, "warm": 0, "cold": 0, "skipped": 0}
batches = {"hot": {"ids":[],"vectors":[],"metadatas":[]},
           "warm":{"ids":[],"vectors":[],"metadatas":[]},
           "cold":{"ids":[],"vectors":[],"metadatas":[]}}

def flush(partition):
    b = batches[partition]
    if not b["ids"]:
        return
    COLLECTIONS[partition].upsert(
        ids=b["ids"],
        embeddings=b["vectors"],
        metadatas=b["metadatas"],
    )
    b["ids"].clear(); b["vectors"].clear(); b["metadatas"].clear()

print(f"Inserting {len(df):,} posts into ChromaDB...")
for i, row in df.iterrows():
    partition = assign_partition(row["created_at"], row["category"])
    if not partition:
        counts["skipped"] += 1
        continue

    vector = build_vector(
        commodity       = row["commodity"],
        target_roles_str= row["target_roles"],
        lat             = row["latitude"],
        lon             = row["longitude"],
        category        = row["category"],
        qty_min         = row.get("qty_min_mt"),
        qty_max         = row.get("qty_max_mt"),
    )

    # ChromaDB metadata — all filter/expiry fields stored here
    meta = {
        "category":       str(row["category"]),
        "commodity":      str(row["commodity"]),
        "target_roles":   str(row["target_roles"]),
        "location_state": str(row["location_state"]),
        "location_city":  str(row["location_city"]),
        "latitude":       float(row["latitude"]),
        "longitude":      float(row["longitude"]),
        "created_at":     str(row["created_at"]),
        "expires_at":     str(row["expires_at"]),
        "is_active":      int(row["is_active"]),
        "partition":      partition,
        # qty & price only for deal_req
        "qty_min_mt":   float(row["qty_min_mt"])   if pd.notna(row.get("qty_min_mt"))   else -1.0,
        "qty_max_mt":   float(row["qty_max_mt"])   if pd.notna(row.get("qty_max_mt"))   else -1.0,
        "price_inr":    float(row["price_per_unit_inr"]) if pd.notna(row.get("price_per_unit_inr")) else -1.0,
    }

    b = batches[partition]
    b["ids"].append(str(row["post_id"]))
    b["vectors"].append(vector)
    b["metadatas"].append(meta)
    counts[partition] += 1

    # flush when batch is full
    if len(b["ids"]) >= BATCH_SIZE:
        flush(partition)

# flush remaining
for p in ["hot","warm","cold"]:
    flush(p)

print("\nLoad complete:")
print(f"  posts_hot  : {counts['hot']:>6,}")
print(f"  posts_warm : {counts['warm']:>6,}")
print(f"  posts_cold : {counts['cold']:>6,}")
print(f"  skipped    : {counts['skipped']:>6,}  (category expired for that age)")
print(f"\nChromaDB counts:")
print(f"  posts_hot  : {posts_hot.count():>6,}")
print(f"  posts_warm : {posts_warm.count():>6,}")
print(f"  posts_cold : {posts_cold.count():>6,}")


## Cell 06 — Expiry job (run hourly via Celery / cron)

In [ ]:
def run_expiry_job(reference_time=None):
    """
    Three steps every hour:

    STEP 1 — Soft-expire: set is_active=0 for posts past expires_at
             (post stays in DB for audit / re-surface logic)

    STEP 2 — Migrate:
             hot  → warm  posts older than 72 hrs still within expiry
             warm → cold  posts older than 5 days still within expiry

    STEP 3 — Hard-delete: remove from cold anything older than 30 days

    To wire into Celery:
        @app.task
        def expiry_task():
            run_expiry_job()

        # In celery beat schedule:
        "expiry-hourly": {
            "task": "tasks.expiry_task",
            "schedule": crontab(minute=0),   # every hour
        }
    """
    now     = reference_time or datetime.now(timezone.utc)
    now_iso = now.isoformat()
    stats   = {"soft_expired":0, "hot_to_warm":0, "warm_to_cold":0, "hard_deleted":0}

    # ── STEP 1: soft-expire across all collections ──────────────────────────
    for name, col in COLLECTIONS.items():
        # get all active posts whose expires_at has passed
        results = col.get(
            where={"$and": [
                {"is_active":  {"$eq": 1}},
                {"expires_at": {"$lte": now_iso}},
            ]},
            include=["metadatas"],
        )
        if results["ids"]:
            for mid, meta in zip(results["ids"], results["metadatas"]):
                meta["is_active"] = 0
                col.update(ids=[mid], metadatas=[meta])
            stats["soft_expired"] += len(results["ids"])

    # ── STEP 2a: migrate hot → warm ─────────────────────────────────────────
    hot_cutoff = (now - timedelta(hours=HOT_HOURS)).isoformat()
    results = posts_hot.get(
        where={"$and": [
            {"is_active":  {"$eq": 1}},
            {"created_at": {"$lte": hot_cutoff}},
        ]},
        include=["embeddings","metadatas"],
    )
    if results["ids"]:
        to_move_ids, to_move_vecs, to_move_metas = [], [], []
        for pid, vec, meta in zip(results["ids"], results["embeddings"], results["metadatas"]):
            if "warm" not in CATEGORY_PARTITIONS.get(meta["category"], []):
                continue  # category not allowed in warm — will be soft-expired
            meta["partition"] = "warm"
            to_move_ids.append(pid)
            to_move_vecs.append(vec)
            to_move_metas.append(meta)

        if to_move_ids:
            posts_warm.upsert(ids=to_move_ids, embeddings=to_move_vecs, metadatas=to_move_metas)
            posts_hot.delete(ids=to_move_ids)
            stats["hot_to_warm"] += len(to_move_ids)

    # ── STEP 2b: migrate warm → cold ────────────────────────────────────────
    warm_cutoff = (now - timedelta(days=WARM_DAYS)).isoformat()
    results = posts_warm.get(
        where={"$and": [
            {"is_active":  {"$eq": 1}},
            {"created_at": {"$lte": warm_cutoff}},
        ]},
        include=["embeddings","metadatas"],
    )
    if results["ids"]:
        to_move_ids, to_move_vecs, to_move_metas = [], [], []
        for pid, vec, meta in zip(results["ids"], results["embeddings"], results["metadatas"]):
            if "cold" not in CATEGORY_PARTITIONS.get(meta["category"], []):
                continue
            meta["partition"] = "cold"
            to_move_ids.append(pid)
            to_move_vecs.append(vec)
            to_move_metas.append(meta)

        if to_move_ids:
            posts_cold.upsert(ids=to_move_ids, embeddings=to_move_vecs, metadatas=to_move_metas)
            posts_warm.delete(ids=to_move_ids)
            stats["warm_to_cold"] += len(to_move_ids)

    # ── STEP 3: hard-delete from cold beyond 30 days ────────────────────────
    cold_cutoff = (now - timedelta(days=COLD_DAYS)).isoformat()
    results = posts_cold.get(
        where={"created_at": {"$lte": cold_cutoff}},
        include=["metadatas"],
    )
    if results["ids"]:
        posts_cold.delete(ids=results["ids"])
        stats["hard_deleted"] += len(results["ids"])

    return stats

# Run once against the loaded data
stats = run_expiry_job(reference_time=CUTOFF)
print("Expiry job complete:")
print(f"  Soft-expired    : {stats['soft_expired']:,}")
print(f"  Moved hot→warm  : {stats['hot_to_warm']:,}")
print(f"  Moved warm→cold : {stats['warm_to_cold']:,}")
print(f"  Hard-deleted    : {stats['hard_deleted']:,}")
print(f"\nPost-expiry counts:")
print(f"  posts_hot  : {posts_hot.count():,}")
print(f"  posts_warm : {posts_warm.count():,}")
print(f"  posts_cold : {posts_cold.count():,}")


## Cell 07 — User profile vector (query-side)

In [ ]:
def build_user_vector(commodity, role, lat, lon,
                      qty_min_mt=None, qty_max_mt=None):
    """
    Build the query vector from the USER's profile.
    Same 18-dim layout as the post vector — cosine similarity
    measures how well a post matches this user.

    For category-specific queries, call this once per category
    with the qty dims zeroed for non-deal retrieval.
    """
    return build_vector(
        commodity        = commodity,
        target_roles_str = role,    # user's own role (is a trader / exporter / broker)
        lat              = lat,
        lon              = lon,
        category         = "deal_req",   # include qty dims
        qty_min          = qty_min_mt,
        qty_max          = qty_max_mt,
    )

# Example: rice trader in Mumbai, trades 50-200 MT
user_vec = build_user_vector(
    commodity  = "rice",
    role       = "trader",
    lat        = 19.0760,
    lon        = 72.8777,
    qty_min_mt = 50,
    qty_max_mt = 200,
)
print(f"User vector ({len(user_vec)} dims): {user_vec}")


## Cell 08 — Query: retrieve posts by user preference

In [ ]:
def query_feed(user_vector, commodity, category=None,
               n_results=30, seen_ids=None):
    """
    Query hot → warm → cold and merge results.

    Args:
        user_vector : 18-dim float list from build_user_vector()
        commodity   : filter to this commodity
        category    : None = all categories, or one of the 4 strings
        n_results   : candidates per partition
        seen_ids    : list of post_ids already shown to this user

    ChromaDB query API:
        collection.query(
            query_embeddings=[user_vector],
            n_results=n_results,
            where={...filters...},
            include=["metadatas","distances"],
        )
    """
    seen = set(seen_ids or [])
    all_results = []

    # ── build where filter
    where_conditions = [
        {"is_active":  {"$eq": 1}},
        {"commodity":  {"$eq": commodity}},
    ]
    if category:
        where_conditions.append({"category": {"$eq": category}})

    where = {"$and": where_conditions} if len(where_conditions) > 1 else where_conditions[0]

    for name, col in COLLECTIONS.items():
        if col.count() == 0:
            continue
        try:
            res = col.query(
                query_embeddings=[user_vector],
                n_results=min(n_results, col.count()),
                where=where,
                include=["metadatas","distances"],
            )
        except Exception:
            continue

        for pid, dist, meta in zip(
            res["ids"][0],
            res["distances"][0],
            res["metadatas"][0],
        ):
            if pid in seen:
                continue
            # ChromaDB cosine distance = 1 - similarity
            similarity = round(1.0 - dist, 4)
            all_results.append({
                "post_id":   pid,
                "score":     similarity,
                "partition": name,
                "category":  meta["category"],
                "commodity": meta["commodity"],
                "roles":     meta["target_roles"],
                "city":      meta["location_city"],
                "state":     meta["location_state"],
                "created_at":meta["created_at"],
                "qty_min":   meta["qty_min_mt"],
                "qty_max":   meta["qty_max_mt"],
                "price_inr": meta["price_inr"],
            })
            seen.add(pid)

    all_results.sort(key=lambda x: x["score"], reverse=True)
    return all_results

print("query_feed() ready.")


## Cell 09 — Demo: rice trader Mumbai

In [ ]:
user_vec = build_user_vector("rice","trader", 19.0760, 72.8777, 50, 200)

results = query_feed(
    user_vector = user_vec,
    commodity   = "rice",
    category    = "deal_req",
    n_results   = 30,
)

print(f"Results for rice trader in Mumbai — deal_req posts\n")
print(f"{'#':<3} {'score':<7} {'part':<6} {'category':<15} {'roles':<22} {'city':<18} {'qty_min':>8} {'qty_max':>8} {'price_inr':>12}")
print("─"*105)
for i, r in enumerate(results[:15], 1):
    qmin = f"{r['qty_min']:.0f}" if r['qty_min'] and r['qty_min'] > 0 else "—"
    qmax = f"{r['qty_max']:.0f}" if r['qty_max'] and r['qty_max'] > 0 else "—"
    price= f"₹{r['price_inr']:,.0f}" if r['price_inr'] and r['price_inr'] > 0 else "—"
    print(f"{i:<3} {r['score']:<7.4f} {r['partition']:<6} {r['category']:<15} "
          f"{r['roles']:<22} {r['city']:<18} {qmin:>8} {qmax:>8} {price:>12}")


## Cell 10 — Demo: cotton exporter Ahmedabad (all categories)

In [ ]:
user_vec_2 = build_user_vector("cotton","exporter", 23.0225, 72.5714)

results_2 = query_feed(
    user_vector = user_vec_2,
    commodity   = "cotton",
    category    = None,    # all categories
    n_results   = 40,
)

print("Results for cotton exporter in Ahmedabad — all categories\n")
print(f"{'#':<3} {'score':<7} {'part':<6} {'category':<15} {'roles':<25} {'city':<15}")
print("─"*80)
for i, r in enumerate(results_2[:20], 1):
    print(f"{i:<3} {r['score']:<7.4f} {r['partition']:<6} {r['category']:<15} {r['roles']:<25} {r['city']:<15}")

# category breakdown
cats = pd.Series([r["category"] for r in results_2]).value_counts()
print(f"\nCategory mix in top {len(results_2)} results:")
print(cats.to_string())


## Cell 11 — Demo: wheat broker Ludhiana, only knowledge posts

In [ ]:
user_vec_3 = build_user_vector("wheat","broker", 30.9010, 75.8573)

results_3 = query_feed(
    user_vector = user_vec_3,
    commodity   = "wheat",
    category    = "knowledge",
    n_results   = 20,
)

print("Results for wheat broker in Ludhiana — knowledge posts only\n")
print(f"{'#':<3} {'score':<7} {'part':<6} {'category':<15} {'roles':<25} {'city':<15} {'created_at'}")
print("─"*100)
for i, r in enumerate(results_3[:15], 1):
    print(f"{i:<3} {r['score']:<7.4f} {r['partition']:<6} {r['category']:<15} {r['roles']:<25} {r['city']:<15} {r['created_at'][:19]}")


## Cell 12 — Score distribution + partition breakdown

In [ ]:
import matplotlib.pyplot as plt

user_vec = build_user_vector("rice","trader", 19.0760, 72.8777, 50, 200)
results  = query_feed(user_vec, "rice", n_results=50)

df_res = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Score histogram
axes[0].hist(df_res["score"], bins=20, color="#3B8BD4", edgecolor="white", linewidth=0.5)
axes[0].set_title("Cosine similarity distribution")
axes[0].set_xlabel("Score"); axes[0].set_ylabel("Count")

# Partition breakdown
part_counts = df_res["partition"].value_counts()
axes[1].bar(part_counts.index, part_counts.values,
            color=["#3B8BD4","#1D9E75","#BA7517"], edgecolor="white")
axes[1].set_title("Results by partition")
axes[1].set_ylabel("Count")

# Category breakdown
cat_counts = df_res["category"].value_counts()
axes[2].bar(cat_counts.index, cat_counts.values,
            color=["#639922","#D85A30","#534AB7","#D4537E"][:len(cat_counts)],
            edgecolor="white")
axes[2].set_title("Results by category")
axes[2].tick_params(axis="x", rotation=20)

plt.suptitle("Rice trader · Mumbai — feed query analysis", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("feed_query_analysis.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: feed_query_analysis.png")


## Cell 13 — Collection health check

In [ ]:
print("── Collection Health ──────────────────────────────────")
for name, col in COLLECTIONS.items():
    total = col.count()
    if total == 0:
        print(f"\n  posts_{name:<4}  EMPTY")
        continue

    sample = col.get(limit=total, include=["metadatas"])
    metas  = sample["metadatas"]
    df_m   = pd.DataFrame(metas)

    active_pct = df_m["is_active"].mean() * 100
    cats       = df_m["category"].value_counts().to_dict()
    comms      = df_m["commodity"].value_counts().head(3).to_dict()

    print(f"\n  posts_{name}")
    print(f"    Total vectors : {total:,}")
    print(f"    Active        : {active_pct:.1f}%")
    print(f"    Categories    : {cats}")
    print(f"    Top commodities: {comms}")
